# Phi-3 Medium Demo - Hadoop vs Spark Comparison

Notebook này chạy trên Google Colab, mount Google Drive để dùng model Phi-3 Medium (nạp dưới dạng 4-bit để tiết kiệm GPU VRAM) để trả lời một câu hỏi duy nhất phân tích và so sánh Hadoop với Apache Spark bằng tiếng Việt với cấu hình System Prompt chuyên gia Big Data.

## 1. Environment and Google Drive Setup

Mount Google Drive để sử dụng Hugging Face cache có sẵn, clone hoặc pull mã nguồn mới nhất từ GitHub và thiết lập thư mục làm việc.

In [1]:
from pathlib import Path
import gc
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/Hutaph/a-triple-of-lms.git"
REPO_BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive")

if IN_COLAB:
    REPO_DIR = Path("/content/a-triple-of-lms")
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    # Tự động phát hiện thư mục root dự án locally
    cwd = Path.cwd()
    if (cwd / "src" / "phi3.py").exists():
        REPO_DIR = cwd
    elif (cwd.parent / "src" / "phi3.py").exists():
        REPO_DIR = cwd.parent
    else:
        REPO_DIR = cwd

if not (REPO_DIR / "src" / "phi3.py").exists():
    raise FileNotFoundError(f"Repo/script not found: {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print("IN_COLAB:", IN_COLAB)
print("Repo:", REPO_DIR)
print("Drive:", DRIVE_ROOT if IN_COLAB else "not mounted")

Mounted at /content/drive
IN_COLAB: True
Repo: /content/a-triple-of-lms
Drive: /content/drive/MyDrive


## 2. Install Dependencies

Cài đặt các gói thư viện cần thiết để nạp và chạy model cục bộ trên Colab GPU.

In [2]:
# Install runtime dependencies. Safe to rerun.
packages = [
    "transformers>=4.41.0",
    "accelerate>=0.30.0",
    "bitsandbytes>=0.43.0",
    "sentencepiece",
    "einops",
    "tqdm",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)
print("Dependencies installed")

Dependencies installed


## 3. Configuration and Prompt Setup

Cấu hình các tham số cho model và thiết lập hệ thống System Prompt và User Prompt bằng tiếng Việt theo đúng yêu cầu.

In [3]:
# Thư mục chứa Hugging Face cache trên Drive
HF_CACHE_DIR = DRIVE_ROOT / "hf_cache"

LOCAL_FILES_ONLY = True       # True = chỉ lấy từ cache trên Drive. Chuyển thành False nếu muốn tải trực tiếp từ Hugging Face
LOAD_IN_4BIT = True           # Nạp dưới dạng 4-bit để phù hợp với Colab GPU
DEVICE = "auto"
TORCH_DTYPE = "float16"

MODEL_ID = "microsoft/Phi-3-medium-4k-instruct"

SYSTEM_PROMPT_VN = (
    "Bạn là một chuyên gia Big Data Engineer có kinh nghiệm triển khai hệ thống xử lý dữ liệu lớn trong môi trường production. "
    "Hãy trả lời chính xác, dễ hiểu, có cấu trúc rõ ràng và liên hệ với các use case thực tế. Khi so sánh công nghệ, "
    "cần nêu rõ kiến trúc, ưu điểm, hạn chế, trường hợp nên sử dụng và trade-off khi triển khai."
)

USER_PROMPT = """Hãy trình bày sự khác nhau giữa Hadoop và Apache Spark trong xử lý dữ liệu lớn.

Yêu cầu câu trả lời bằng tiếng Việt, có cấu trúc rõ ràng theo các phần sau:

1. Tổng quan ngắn gọn về Hadoop và Apache Spark.
2. So sánh hai công nghệ theo các tiêu chí:
     - Kiến trúc xử lý dữ liệu
     - Cơ chế lưu trữ và tính toán
     - Hiệu năng
     - Khả năng xử lý batch, streaming và interactive analytics
     - Mức độ phù hợp trong các hệ thống Big Data hiện đại
3. Trình bày ưu điểm và hạn chế của Hadoop.
4. Trình bày ưu điểm và hạn chế của Apache Spark.
5. Cho ví dụ thực tế: khi nào nên dùng Hadoop, khi nào nên dùng Spark.
6. Kết luận ngắn gọn: nếu xây dựng một pipeline phân tích dữ liệu lớn hiện nay, nên chọn công nghệ nào trong từng trường hợp.

Yêu cầu thêm:
    - Trả lời dễ hiểu cho sinh viên mới học Big Data.
    - Không chỉ liệt kê, hãy giải thích ý nghĩa của từng điểm khác biệt.
    - Có thể dùng bảng so sánh nếu phù hợp.
    - Không trả lời quá dài, ưu tiên rõ ràng và thực tế."""

## 4. Run Generation

Nạp model sử dụng logic của phi3.py và sinh câu trả lời duy nhất.

In [5]:
from src.phi3 import (
    normalize_hub_cache_dir,
    run_single_inference,
)

cache_dir = normalize_hub_cache_dir(str(HF_CACHE_DIR)) if HF_CACHE_DIR else None
print("Resolved cache_dir:", cache_dir)

print("Starting inference (it may take a few minutes)...\n")
response = run_single_inference(
    model_id_or_path=MODEL_ID,
    prompt=USER_PROMPT,
    system_prompt=SYSTEM_PROMPT_VN,
    cache_dir=cache_dir,
    local_files_only=LOCAL_FILES_ONLY,
    device=DEVICE,
    torch_dtype_name=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    trust_remote_code=False,
    temperature=0.2,
    max_new_tokens=6000,
)

Resolved cache_dir: /content/drive/MyDrive/hf_cache/hub
Starting inference (it may take a few minutes)...



Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generated response in 254.00s (Prompt tokens: 571, Completion tokens: 1840)


## 5. Render Response

Hiển thị câu trả lời có định dạng Markdown hoàn chỉnh và đẹp mắt.

In [6]:
from IPython.display import display, Markdown

print("Câu trả lời từ Phi-3 Medium:\n")
display(Markdown(response))

Câu trả lời từ Phi-3 Medium:



1. Tổng quan ngắn gọn về Hadoop và Apache Spark:

Hadoop là một hệ thống Big Data phát triển bởi Apache, thu hút sự đồng bội của nhiều công ty và nhà nghiên cứu. Nó được sản xuất bởi Hortonworks và nửa đời của nó bao gồm các thuật ngữ như HDFS (Hadoop Distributed File System), MapReduce và YARN (Yet Another Resource Negotiator).

Apache Spark là một hệ thống Big Data thu hút sự đồng bội của Hadoop, nhưng nó được sản xuất bởi Apache. Nó được đặc trưng bởi sự điều chỉnh tốt và đầu khích cho các giao thức với đầu tư đầu khích, như SQL, streaming và machine learning.

2. So sánh hai công nghệ theo các tiêu chính:

- Kiến trúc xử lý dữ liệu:
  Hadoop được sử dụng với 3-tier kiến trúc: HDFS, MapReduEE, YARN.
  Spark được sử dụng với 2-tier kiến trúc: Spark Core và Spark SQL.

- Cơ chế lưu trữ và tính toán:
  Hadoop sử dụng HDFS để lưu trữ dữ liệu và MapReduce để tính toán.
  Spark sử dụng RDD (Resilient Distributed Dataset) và đồng hợp đầu khích để lưu trữ và tính toán.

- Hiệu năng:
  Hadoop chỉ được sử dụng cho phần tính toán, với đầu tư đầu khích và đầu tư đầu khích.
  Spark được sử dụng cho tất cả các giao thức, với đầu tư đầu khích, đầu tư đầu khích và đầu tư đầu khích.

- Khả năng xử lý batch, streaming và interactive analytics:
  Hadoop chỉ được sử dụng cho phần xử lý batch, với đầu tư đầu khích và đầu tư đầu khích.
  Spark được sử dụng cho tất cả các giao thức, với đầu tư đầu khích, đầu tư đầu khích và đầu tức.

- Mức độ phù hợp trong các hệ thống Big Data hiện đại:
  Hadoop được sử dụng với các hệ thống Big Data hiện đại như Cloudera, Hortonworks và MapR.
  Spark được sử dụng với các hệ thống Big Data hiện đại như Cloudera, Hortonworks và Databricks.

3. Được ưu điểm và hạn chế của Hadoop:

- Được ưu điểm:
  Hadoop được sử dụng với các dữ liệu lớn, với đầu tức và đầu tức đầu khích.
  Hadoop được sử dụng với các hệ thống Big Data hiện đại như Cloudera, Hortonworks và MapR.

- Hạn chế:
  Hadoop chỉ được sử dụng cho phần tính toán, với đầu tức và đầu tức đầu khích.
  Hadoop chỉ được sử dụng cho phần xử lý batch, với đầu tức và đầu tức đầu khích.

4. Được ưu điểm và hạn chế của Apache Spark:

- Được ưu điểm:
  Spark được sử dụng với các giao thức, với đầu tức và đầu tức đầu khích.
  Spark được sử dụng với các hệ thống Big Data hiện đại như Cloudera, Hortonworks và Databricks.

- Hạn chế:
  Spark chỉ được sử dụng với các hệ thống Big Data hiện đại như Cloudera, Hortonworks và Databricks.

5. Cho ví dụ thực tế:

- Khi nào nên dùng Hadoop:
  Hadoop chỉ được sử dụng cho phần xử lý batch, với các dữ liệu lớn và đầu tức và đầu tức đầu khích.

- Khi nào nên dùng Spark:
  Spark được sử dụng cho tất cả các giao thức, với các dữ liệu lớn và đầu tức và đầu tức đầu khích.

6. Kết luận ngắn gọn:

Nếu xây dựng một pipeline phân tích dữ liệu lớn hiện đại, nên chọn Hadoop khi nên dùng để xử lý batch và đầu tức và đầu tức đầu khích. Nếu nên dùng Spark, nên chọn để xử lý tất cả các giao thức, với đầu tức và đầu tức đầu khích.